In [1]:
import duckdb
import pandas as pd
from datetime import date, timedelta

# 00 - Criando um DataFrame

In [7]:
data = {
    "id": list(range(1, 11)),
    "value": [10.5, 20.1, 30.2, 40.8, 50.0, 60.3, 70.7, 80.9, 90.4, 100.6],
    "name": ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"],
    "active": [True, False, True, True, False, True, False, True, False, True],
    "date": [date(2024, 1, 1) + timedelta(days=i) for i in range(10)]
}

In [8]:
df = pd.DataFrame(data)

In [9]:
conn = duckdb.connect()
conn.execute("CREATE TEMP TABLE minha_tabela AS SELECT * FROM df")

In [15]:
# Tipos das colunas
conn.execute("DESCRIBE minha_tabela").fetchdf()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,value,DOUBLE,YES,None,None,None
2,name,VARCHAR,YES,None,None,None
3,active,BOOLEAN,YES,None,None,None
4,date,DATE,YES,None,None,None


In [16]:
conn.execute("DESCRIBE minha_tabela").fetchdf()[["column_type"]]

,column_type
0,BIGINT
1,DOUBLE
2,VARCHAR
3,BOOLEAN
4,DATE


In [5]:
# Executar consultas
resultado = conn.execute("SELECT * FROM minha_tabela WHERE active = true").fetchdf()
print(resultado)

   id  value name  active       date
0   1   10.5    A    True 2024-01-01
1   3   30.2    C    True 2024-01-03
2   4   40.8    D    True 2024-01-04
3   6   60.3    F    True 2024-01-06
4   8   80.9    H    True 2024-01-08
5  10  100.6    J    True 2024-01-10


In [6]:
resultado = conn.execute("SELECT * FROM df WHERE value > 50").fetchdf()
print(resultado)

   id  value name  active       date
0   6   60.3    F    True 2024-01-06
1   7   70.7    G   False 2024-01-07
2   8   80.9    H    True 2024-01-08
3   9   90.4    I   False 2024-01-09
4  10  100.6    J    True 2024-01-10


# 01 - Lendo um Arquivo

In [31]:
conn = duckdb.connect()
df = conn.sql("""
SELECT *
FROM 'data/data_raw.csv'
""")

In [32]:
df.limit(5).show()

┌───────┬─────────────┬──────────────┬───────────────┬────────────┬────────────────────┬─────────┬───────────┐
│ sales │ customer_id │ customer_age │ purchase_date │  product   │ last_purchase_date │  price  │ seller_id │
│ int64 │    int64    │    int64     │     date      │  varchar   │        date        │ double  │   int64   │
├───────┼─────────────┼──────────────┼───────────────┼────────────┼────────────────────┼─────────┼───────────┤
│    52 │        1511 │           86 │ 2024-10-07    │ Laptop     │ 2022-05-29         │ 2534.07 │       199 │
│    93 │        4985 │           52 │ 2023-12-10    │ Smartphone │ 2023-09-20         │ 1879.78 │       182 │
│    15 │        3092 │           69 │ 2024-01-06    │ TV         │ 2022-07-06         │ 4844.32 │       112 │
│    72 │        4179 │           31 │ 2023-02-02    │ Smartphone │ 2022-01-20         │ 4705.96 │       147 │
│    61 │        3266 │           80 │ 2024-01-09    │ Tablet     │ 2023-01-08         │ 4766.52 │       159 │
└